In [1]:
# # VADER Sentiment Analysis — NPS Data
# 
# VADER is a sentiment analysis tool that reads text and tells you if it's positive, negative, or neutral.
# It scores each response from -1 (very negative) to +1 (very positive). No training needed — it just works.
#
# I'm using it here to check if the text customers wrote matches what the EDA showed numerically.
# The EDA flagged Reliability, Performance, and Expectation Gap as problem areas — let's see if the words agree.


# --- CODE CELL ---
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from langdetect import detect, LangDetectException

analyzer = SentimentIntensityAnalyzer()


# --- MARKDOWN CELL ---
# ## Load the data
# Pulling in the cleaned NPS dataset. The columns we care about are Response Text, Translated Text, and Taxonomy Type.


# --- CODE CELL ---
df = pd.read_csv("/Users/Apple/Desktop/Hubspot/data/processed/nps.csv")
print(f"Loaded {len(df)} rows")
df[["Response Text", "Translated Text", "Taxonomy Type"]].head(10)




Loaded 11440 rows


,Response Text,Translated Text,Taxonomy Type
0,Seems to be an excessive amount of repeating s...,NaN,NaN
1,price,NaN,NaN
2,it's handy,NaN,NaN
3,æŽ¥ç¶šé€Ÿåº¦ãŒé…ã„,NaN,NaN
4,Still find it hard to get it to do what I want...,NaN,NaN
5,Es sencillo de entender,NaN,NaN
6,I like the UX; it's easy to use and learn. Is ...,NaN,NaN
7,AI is bad,NaN,NaN
8,"Bastante amplio, prÃ¡ctico, Ãºtil y sobre todo...",NaN,NaN
9,Easier to navigate,NaN,NaN


In [2]:
 #--- MARKDOWN CELL ---
# ## Pick which text to score
# Some responses aren't in English, so:
# - If there's a translation, use that
# - If not, check if the original is English and use it
# - Otherwise skip it (mark as "unscored")


# --- CODE CELL ---
def is_english(text):
    try:
        return detect(str(text)) == "en"
    except:
        return False

def get_text_to_score(row):
    translated = row.get("Translated Text")
    original = row.get("Response Text")

    if pd.notna(translated) and str(translated).strip():
        return str(translated).strip()

    if pd.notna(original) and str(original).strip():
        if is_english(original):
            return str(original).strip()

    return None

df["text_to_score"] = df.apply(get_text_to_score, axis=1)

print(f"Scoreable: {df['text_to_score'].notna().sum()}")
print(f"Unscored: {df['text_to_score'].isna().sum()}")

Scoreable: 9776
Unscored: 1664


In [3]:
# --- MARKDOWN CELL ---
# ## Run VADER
# Each response gets a compound score. Then I label it:
# - Above 0.05 → positive
# - Below -0.05 → negative
# - In between → neutral


# --- CODE CELL ---
def analyze_sentiment(text):
    if text is None or pd.isna(text):
        return pd.Series({"vader_compound": None, "sentiment_label": "unscored"})

    scores = analyzer.polarity_scores(text)

    if scores["compound"] >= 0.05:
        label = "positive"
    elif scores["compound"] <= -0.05:
        label = "negative"
    else:
        label = "neutral"

    return pd.Series({"vader_compound": scores["compound"], "sentiment_label": label})

results = df["text_to_score"].apply(analyze_sentiment)
df = pd.concat([df, results], axis=1)

df[["Response Text", "vader_compound", "sentiment_label"]].head(10)

,Response Text,vader_compound,sentiment_label
0,Seems to be an excessive amount of repeating s...,0.0000,neutral
1,price,NaN,unscored
2,it's handy,0.0000,neutral
3,æŽ¥ç¶šé€Ÿåº¦ãŒé…ã„,NaN,unscored
4,Still find it hard to get it to do what I want...,0.0516,positive
5,Es sencillo de entender,NaN,unscored
6,I like the UX; it's easy to use and learn. Is ...,0.6597,positive
7,AI is bad,NaN,unscored
8,"Bastante amplio, prÃ¡ctico, Ãºtil y sobre todo...",NaN,unscored
9,Easier to navigate,0.4215,positive


In [5]:
# --- MARKDOWN CELL ---
# ## Sentiment by Taxonomy Type
# This is the main output — how many positive vs negative responses in each product area.


# --- CODE CELL ---
scored_df = df[df["sentiment_label"] != "unscored"]

summary = scored_df.groupby(["Taxonomy Type", "sentiment_label"]).size().unstack(fill_value=0)

for col in ["positive", "negative", "neutral"]:
    if col not in summary.columns:
        summary[col] = 0

summary = summary[["positive", "negative", "neutral"]]
summary["total"] = summary.sum(axis=1)
summary["positive_pct"] = (summary["positive"] / summary["total"] * 100).round(1)
summary["negative_pct"] = (summary["negative"] / summary["total"] * 100).round(1)
summary["net_sentiment"] = (summary["positive_pct"] - summary["negative_pct"]).round(1)
summary = summary.sort_values("net_sentiment", ascending=False)

summary

sentiment_label,positive,negative,neutral,total,positive_pct,negative_pct,net_sentiment
Taxonomy Type,,,,,,,
General,1269,283,613,2165,58.6,13.1,45.5
Usability,2714,1019,1031,4764,57.0,21.4,35.6
Functionality,687,311,412,1410,48.7,22.1,26.6
Expectation Gap,144,116,153,413,34.9,28.1,6.8
Performance,105,125,303,533,19.7,23.5,-3.8
Reliability,81,207,183,471,17.2,43.9,-26.7


In [6]:
# --- MARKDOWN CELL ---
# ## Overall numbers


# --- CODE CELL ---
total = len(scored_df)
print(f"Positive:  {(scored_df['sentiment_label'] == 'positive').sum()} ({(scored_df['sentiment_label'] == 'positive').sum()/total*100:.1f}%)")
print(f"Negative:  {(scored_df['sentiment_label'] == 'negative').sum()} ({(scored_df['sentiment_label'] == 'negative').sum()/total*100:.1f}%)")
print(f"Neutral:   {(scored_df['sentiment_label'] == 'neutral').sum()} ({(scored_df['sentiment_label'] == 'neutral').sum()/total*100:.1f}%)")
print(f"Unscored:  {df['text_to_score'].isna().sum()}")
print(f"Total:     {len(df)}")

Positive:  5014 (51.3%)
Negative:  2063 (21.1%)
Neutral:   2699 (27.6%)
Unscored:  1664
Total:     11440


In [7]:
# --- MARKDOWN CELL ---
# ## Does the text match what the EDA found?
#
# | EDA (numeric NPS) | VADER (text sentiment) | Match? |
# |---|---|---|
# | Reliability worst NPS (-65) | Worst net sentiment (-26.9), 44% negative | ✅ |
# | Performance critical (-69) | Net negative (-3.3), mostly neutral/negative | ✅ |
# | Expectation Gap negative (-63) | Barely positive (+6.8), nearly even split | ✅ |
# | General & Usability are strengths | Strongest sentiment (+45.4, +35.6) | ✅ |
#
# The text confirms the numbers. Customers writing about Reliability and Performance are genuinely frustrated — it's not just low scores, the words back it up.


# --- CODE CELL ---
# save everything
df.drop(columns=["text_to_score"]).to_csv("nps_sentiment_results.csv", index=False)
summary.to_csv("nps_sentiment_summary.csv")
print("Saved: nps_sentiment_results.csv & nps_sentiment_summary.csv")




Saved: nps_sentiment_results.csv & nps_sentiment_summary.csv
